# Lesson 24: Does Music Make You Faster?

A running coach claims that listening to music makes runners faster on a 2-mile run. She **randomly assigns** 26 volunteers: 14 run with music, 12 run without.

Let $\mu_1$ = mean time with music and $\mu_2$ = mean time without. "Faster" means *lower* times, so:

$$H_0: \mu_1 - \mu_2 = 0 \qquad H_a: \mu_1 - \mu_2 < 0$$

Run each cell in order (Shift + Enter).

> Colab setup: *Runtime → Change runtime type → R*.

## 1. The data

In [ ]:
library(ggplot2)
set.seed(206)
options(repr.plot.width = 12, repr.plot.height = 7, repr.plot.res = 120)

# 2-mile times (minutes) for runners randomly assigned to each group
music   <- c(13.6, 13.7, 14.0, 14.4, 14.7, 14.7, 14.7, 15.1, 15.1, 15.3, 15.4, 15.7, 15.9, 16.3)
nomusic <- c(13.7, 14.5, 15.1, 15.2, 15.6, 15.6, 15.9, 16.5, 16.7, 16.8, 17.3, 17.3)

runs <- data.frame(time  = c(music, nomusic),
                   group = rep(c("Music", "No music"), c(length(music), length(nomusic))))

cat("Music:    n =", length(music),   "  mean =", round(mean(music), 2),   "  s =", round(sd(music), 3), "\n")
cat("No music: n =", length(nomusic), "  mean =", round(mean(nomusic), 2), "  s =", round(sd(nomusic), 3), "\n")
cat("Difference in means (music - no music):", round(mean(music) - mean(nomusic), 2), "minutes\n")

ggplot(runs, aes(time, group, color = group)) +
  geom_point(size = 5, alpha = 0.7, position = position_jitter(width = 0, height = 0.08)) +
  stat_summary(fun = mean, geom = "point", shape = 124, size = 18, color = "black") +
  scale_color_manual(values = c("Music" = "navy", "No music" = "darkorange"), guide = "none") +
  labs(title = "2-mile times: music vs. no music",
       subtitle = "Black bars mark each group's mean. The groups overlap a lot.",
       x = "2-mile time (minutes)", y = NULL) +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** the music group averaged about a minute faster, but the two groups overlap a lot. Some no-music runners beat most of the music group. Is a 0.95-minute gap real, or could it be the luck of who landed in which group?

## 2. What if music did nothing?

If music has no effect, each runner would have run the same time in either group. The labels "Music" and "No music" would be meaningless. So we can **shuffle the labels**, deal the same 26 times into new groups of 14 and 12, and see what difference chance alone produces.

In [ ]:
# If music does NOTHING, the labels "Music" and "No music" are meaningless.
# Shuffle the labels and see what difference we get by pure chance.
observed_diff <- mean(music) - mean(nomusic)

shuffle_once <- function() {
  shuffled <- sample(runs$group)                     # deal the same 26 times into random groups
  mean(runs$time[shuffled == "Music"]) - mean(runs$time[shuffled == "No music"])
}

for (i in 1:5) cat("Shuffle", i, ": difference =", round(shuffle_once(), 2), "minutes\n")
cat("Real experiment: difference =", round(observed_diff, 2), "minutes\n")

**What to notice:** shuffled differences bounce around zero, sometimes positive, sometimes negative. None of these five came close to $-0.95$.

## 3. Shuffle 10,000 times

Now do it 10,000 times. The fraction of shuffles at least as extreme as the real result is a P-value: *if music did nothing, how often would luck alone produce a gap this big?*

In [ ]:
# Do it 10,000 times: the differences chance alone produces
reps <- 10000
shuffles <- data.frame(diff = replicate(reps, shuffle_once()))
p_shuffle <- mean(shuffles$diff <= observed_diff)

ggplot(shuffles, aes(diff, fill = diff <= observed_diff)) +
  geom_histogram(binwidth = 0.05, boundary = observed_diff, color = "white") +
  scale_fill_manual(values = c(`TRUE` = "firebrick", `FALSE` = "grey70"), guide = "none") +
  geom_vline(xintercept = observed_diff, linewidth = 1.2) +
  annotate("text", x = 0.6, y = Inf, vjust = 1.5, hjust = 0, size = 5.5,
           label = paste0("real experiment: ", round(observed_diff, 2), " min\n",
                          "shuffles this extreme: ", round(100 * p_shuffle, 1), "%")) +
  labs(title = "10,000 shuffles: what if music did nothing?",
       subtitle = "Red = shuffles where the 'music' group was at least 0.95 minutes faster by luck",
       x = "difference in means, music - no music (minutes)", y = "number of shuffles") +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** only about 1% of shuffles made the "music" group 0.95 minutes faster by luck. A result this extreme is rare if music does nothing.

## 4. The two-sample $t$ test

The $t$ test gets nearly the same answer with a formula instead of shuffling:

$$t = \frac{\bar{x} - \bar{y}}{\sqrt{\dfrac{s_1^2}{m} + \dfrac{s_2^2}{n}}}$$

The degrees of freedom come from a messier formula (it's on the boardsheet). R computes it for you.

In [ ]:
# The two-sample t test, by hand
m <- length(music); n <- length(nomusic)
v1 <- var(music) / m          # s1^2 / m
v2 <- var(nomusic) / n        # s2^2 / n
se <- sqrt(v1 + v2)
t_stat <- (mean(music) - mean(nomusic)) / se
df <- (v1 + v2)^2 / (v1^2 / (m - 1) + v2^2 / (n - 1))

cat("standard error =", round(se, 3), "\n")
cat("test statistic =", round(t_stat, 2), "\n")
cat("degrees of freedom =", round(df, 2), " (round down to", floor(df), "for the table)\n")
cat("P-value from t:", round(pt(t_stat, df), 4), "   P-value from shuffling:", round(p_shuffle, 4), "\n")

# R does all of it in one line
t.test(music, nomusic, alternative = "less")

**What to notice:** the shuffle P-value and the $t$-test P-value agree closely (about 0.01). Two completely different methods, same conclusion: **reject $H_0$**. R's `df = 19.616` matches the hand calculation; round down to 19 when using the table.

## 5. Why the standard error adds the variances

Each group's average wobbles from sample to sample. The **difference** of two averages has two sources of wobble, so it wobbles more. Below, music does nothing and every runner comes from the same population.

In [ ]:
# Why the standard error ADDS the two variances.
# Imagine music does nothing: every runner's time comes from the same population.
xbar_music   <- replicate(reps, mean(rnorm(m, 15.4, 1)))
xbar_nomusic <- replicate(reps, mean(rnorm(n, 15.4, 1)))

spread <- data.frame(
  value = c(xbar_music, xbar_nomusic, xbar_music - xbar_nomusic),
  what  = factor(rep(c("Music average", "No-music average", "Difference of the two"), each = reps),
                 levels = c("Music average", "No-music average", "Difference of the two")))

cat("SD of music averages:     ", round(sd(xbar_music), 3), "\n")
cat("SD of no-music averages:  ", round(sd(xbar_nomusic), 3), "\n")
cat("SD of the difference:     ", round(sd(xbar_music - xbar_nomusic), 3),
    "  (formula: sqrt(1/14 + 1/12) =", round(sqrt(1/m + 1/n), 3), ")\n")

ggplot(spread, aes(value - ifelse(what == "Difference of the two", 0, 15.4))) +
  geom_histogram(binwidth = 0.05, fill = "steelblue", color = "white") +
  facet_wrap(~ what, ncol = 1) +
  labs(title = "Each average wobbles. Their difference wobbles MORE.",
       subtitle = "Two sources of chance add up, so the variances add: SE = sqrt(s1^2/m + s2^2/n)",
       x = "distance from the center (minutes)", y = "count") +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** the bottom histogram is wider than either one above it. Variances add: $\text{SD}(\bar{X} - \bar{Y}) = \sqrt{\sigma^2/m + \sigma^2/n}$. That's why the standard error has a **plus** sign under the square root, even though we're subtracting the means.

## 6. Confidence interval for $\mu_1 - \mu_2$

The confidence interval tells us how big the effect is, not just whether it exists. A confidence interval is two-sided, so we use R's default (no `alternative`).

In [ ]:
# 95% confidence interval for mu1 - mu2 (two-sided, so use R's default alternative)
ci <- t.test(music, nomusic)$conf.int
cat("95% CI for mu1 - mu2:", round(ci[1], 2), "to", round(ci[2], 2), "minutes\n")

options(repr.plot.height = 3.5)
ggplot() +
  annotate("segment", x = ci[1], xend = ci[2], y = 1, yend = 1, linewidth = 2, color = "navy") +
  annotate("segment", x = ci, xend = ci, y = 0.85, yend = 1.15, linewidth = 2, color = "navy") +
  geom_point(aes(x = observed_diff, y = 1), size = 5, color = "navy") +
  geom_vline(xintercept = 0, linetype = "dashed", linewidth = 1) +
  annotate("text", x = 0.02, y = 1.3, hjust = 0, size = 5, label = "0 = no difference") +
  scale_y_continuous(limits = c(0.5, 1.5), breaks = NULL) +
  scale_x_continuous(limits = c(-2, 0.7)) +
  labs(title = "95% confidence interval for the effect of music",
       subtitle = "The whole interval is below 0: music lowers the mean time",
       x = "music - no music (minutes)", y = NULL) +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

**What to notice:** we're 95% confident music lowers the mean 2-mile time by between about 0.14 and 1.76 minutes. Zero isn't in the interval, which agrees with the test. Because runners were **randomly assigned**, we can say music *causes* the improvement.

## 7. Our own experiment: anchoring

Half the room saw "taller or shorter than **1,200 feet**?" and half saw "**85 feet**?" before guessing the tallest tree's height. **Replace the example numbers below with your section's guesses**, then run the cell.

In [ ]:
options(repr.plot.height = 7)
# REPLACE these example guesses with your section's data (feet)
high <- c(350, 400, 450, 500, 600, 650, 700, 800, 900, 1100)   # saw 1,200 ft
low  <- c(80, 100, 120, 150, 150, 180, 200, 250, 300, 400)      # saw 85 ft

guesses <- data.frame(guess  = c(high, low),
                      anchor = rep(c("High anchor (1,200 ft)", "Low anchor (85 ft)"),
                                   c(length(high), length(low))))

ggplot(guesses, aes(guess, anchor, color = anchor)) +
  geom_point(size = 5, alpha = 0.7, position = position_jitter(width = 0, height = 0.08)) +
  stat_summary(fun = mean, geom = "point", shape = 124, size = 18, color = "black") +
  geom_vline(xintercept = 380, linetype = "dashed") +
  annotate("text", x = 390, y = 2.45, hjust = 0, size = 5, label = "real answer: about 380 ft") +
  scale_color_manual(values = c("firebrick", "steelblue"), guide = "none") +
  labs(title = "One irrelevant number moved everyone's guess",
       subtitle = "Black bars mark each group's mean guess",
       x = "guessed height (feet)", y = NULL) +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

t.test(high, low, alternative = "greater")

**What to notice:** check the dotplot before trusting the test. Guesses like these are often right-skewed, with one or two wild guesses. With only about 10 per group, a single 10,000-foot guess can wreck the $t$ test. Everyone knew the anchor was meaningless, and it still moved the guesses.

## 8. Reading R output: the Taylor Swift effect

Travis Kelce's receiving yards in every 2023 game he played, split by whether Taylor Swift attended. Fans claimed he plays better when she's there.

In [ ]:
swift    <- c(16, 28, 44, 60, 69, 71, 75, 81, 83, 93, 116, 124, 179)
no_swift <- c(14, 26, 44, 58, 67, 91)

kelce <- data.frame(yards = c(swift, no_swift),
                    group = rep(c("Swift attended", "Swift absent"), c(length(swift), length(no_swift))))

ggplot(kelce, aes(yards, group, color = group)) +
  geom_point(size = 5, alpha = 0.7, position = position_jitter(width = 0, height = 0.08)) +
  stat_summary(fun = mean, geom = "point", shape = 124, size = 18, color = "black") +
  annotate("text", x = 178, y = 2.3, hjust = 1, size = 5, label = "179 yds vs. Chargers") +
  scale_color_manual(values = c("Swift attended" = "purple", "Swift absent" = "grey50"), guide = "none") +
  labs(title = "Travis Kelce's receiving yards, 2023",
       subtitle = "Black bars mark each group's mean",
       x = "receiving yards", y = NULL) +
  theme_minimal(base_size = 16) +
  theme(plot.background = element_rect(fill = "white", color = NA))

t.test(swift, no_swift, alternative = "greater")

**What to notice:** P = 0.045, just under 0.05. Now drop the one huge game and rerun.

In [ ]:
swift_no_179 <- swift[swift != 179]
t.test(swift_no_179, no_swift, alternative = "greater")

**What to notice:** remove one game and the P-value jumps to 0.084: fail to reject. With samples this small, one big day decides the test.

And even the original "reject" can't show that Swift **causes** anything. She chose which games to attend: all four playoff games are in her group, and most of the games she missed were on the road. Compare that with the music and anchoring experiments, where **random assignment** made the groups alike in everything except the treatment.

### Questions to explore
1. In Section 2, why is it fair to shuffle the labels? What would be wrong with shuffling if runners had *chosen* their group?
2. Add `m <- 100; n <- 100` as the first line of the Section 5 cell and rerun it. What happens to the spread of the difference?
3. In Section 7, add one wild guess of 10000 to the `high` group and rerun. What happens to $s_1$, the test statistic, and the P-value? Does the dotplot still look like a fit for the $t$ test?
4. In Section 8, the fans picked a one-sided test *after* seeing Kelce's numbers. Why does choosing the direction after looking at the data make a small P-value easier to get?
5. Try `t.test(music, nomusic, paired = TRUE)`. R gives an error. Why? What would have to be true about the design for a paired test to make sense?